In [51]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_iris
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [52]:
X, y = load_iris(return_X_y=True)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
x_train, x_val, y_train, y_val = train_test_split(X_scaled, y, test_size = 1./3, random_state=42, shuffle=True)


In [53]:
x_train = torch.tensor(x_train, dtype=torch.float32)
x_val = torch.tensor(x_val, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.long)
train_ds = TensorDataset(x_train, y_train)
val_ds = TensorDataset(x_val, y_val)
train_dl = DataLoader(dataset=train_ds, batch_size=16, shuffle=True, drop_last=True)
val_dl = DataLoader(dataset=val_ds, batch_size=64, shuffle=False, drop_last=False)

In [54]:
class MLPModel(torch.nn.Module):
  def __init__(self, input, hidden, output):
    super().__init__()
    self.fc1 = torch.nn.Linear(input, hidden)
    self.fc2 = torch.nn.Linear(hidden, output)
  def forward(self, x):
    x = self.fc1(x)
    x = torch.relu(x)
    x = self.fc2(x)
    return x

In [55]:
model = MLPModel(input=x_train.shape[1], hidden=8, output=3)

In [56]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

In [57]:
epoches = 400
log_epoche = 50
for epoche in range(epoches):
  model.train()
  for x_batch, y_batch in train_dl:
    y_pred = model(x_batch)
    loss = loss_fn(y_pred, y_batch)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
  model.eval()
  for x_val, y_val in val_dl:
      y_pred = model(x_val)
      loss_val = loss_fn(y_pred, y_val)
  if epoche % log_epoche == 0:
    print(f'epoche: {epoche}, train loss: {loss.item():.4f}, val_loss: {loss_val.item():.4f}')

epoche: 0, train loss: 0.9373, val_loss: 0.9332
epoche: 50, train loss: 0.4065, val_loss: 0.3989
epoche: 100, train loss: 0.2454, val_loss: 0.2288
epoche: 150, train loss: 0.1568, val_loss: 0.1414
epoche: 200, train loss: 0.1767, val_loss: 0.0967
epoche: 250, train loss: 0.1196, val_loss: 0.0771
epoche: 300, train loss: 0.0779, val_loss: 0.0666
epoche: 350, train loss: 0.0346, val_loss: 0.0617


In [58]:
logits = model(x_val)
preds = logits.argmax(dim=1)

acc = accuracy_score(
    y_val.cpu().numpy(),
    preds.cpu().numpy()
)


In [59]:
acc

0.98